# Projet Kayak — Partie 4 : Fusion et visualisation finale

On combine les deux sources de données :
- **Météo** (API OpenWeatherMap) → 35 villes scorées sur 5 jours
- **Hôtels** (scraping Booking via Scrapy + Playwright) → 100 hôtels (20 × 5 villes du top météo)

Et on construit les visualisations qui répondent à la question business :
**"Quelles sont les meilleures destinations cette semaine, et où loger ?"**

## 1. Chargement des données

In [1]:
import pandas as pd
import plotly.express as px

# Données météo des 35 villes
df_weather = pd.read_csv("data/weather.csv")

# Données hôtels des 5 meilleures villes (format JSON Lines)
df_hotels = pd.read_json("data/hotels.jsonl", lines=True)

print(f"Météo  : {len(df_weather)} villes")
print(f"Hôtels : {len(df_hotels)} hôtels sur {df_hotels['city'].nunique()} villes")

Météo  : 35 villes
Hôtels : 100 hôtels sur 5 villes



## 2. Carte des hôtels colorés par note

Vue dédiée des 100 hôtels scrapés. Couleur = note Booking, taille = note (visuel renforcé).

Les hôtels sans note sont exclus (ils n'apportent pas d'info pour la décision).

In [2]:
# On exclut les hôtels sans note pour la viz (10/100)
df_hotels_scored = df_hotels.dropna(subset=["score"]).copy()
print(f"Hôtels affichés : {len(df_hotels_scored)} / {len(df_hotels)} (les autres n'ont pas de note Booking)")

Hôtels affichés : 90 / 100 (les autres n'ont pas de note Booking)


In [3]:
fig_hotels = px.scatter_map(
    df_hotels_scored,
    lat="lat",
    lon="lon",
    color="score",                        # rouge → vert : note basse → haute
    size="score",                         # taille proportionnelle (toutes positives, pas de souci)
    size_max=18,
    hover_name="name",                    # nom de l'hôtel en gras au survol
    hover_data={
        "city": True,
        "score": ":.1f",
        "url": False,                     # URL trop longue dans le tooltip
        "lat": False,
        "lon": False,
    },
    color_continuous_scale="RdYlGn",      # rouge / jaune / vert
    range_color=[6, 10],                  # on cadre l'échelle pour mieux différencier les notes
    zoom=4.5,
    center={"lat": 46.6, "lon": 2.5},
    height=600,
    title="100 hôtels Booking dans les 5 meilleures destinations météo",
)
fig_hotels.update_layout(map_style="carto-positron")
fig_hotels.show()

## 3. Carte croisée météo + hôtels (livrable principal)

On fusionne les deux datasets dans une carte unique. Pour ça on doit :
1. Récupérer le score météo de chaque ville scrapée
2. Construire un **score combiné** par hôtel = mélange de la note Booking et du score météo de sa ville

C'est ce score combiné qui ressort comme la "recommandation finale" de la plateforme.

In [4]:
# Jointure : on rattache le score météo à chaque hôtel via la colonne 'city'
df_merged = df_hotels_scored.merge(
    df_weather[["city", "weather_score", "temp_avg", "rain_total"]],
    on="city",
    how="left",
)

# Score combiné : moyenne pondérée note hôtel (0-10) et score météo (variable)
# On normalise le score météo entre 0 et 10 pour pouvoir les combiner
min_w, max_w = df_merged["weather_score"].min(), df_merged["weather_score"].max()
df_merged["weather_score_norm"] = (
    (df_merged["weather_score"] - min_w) / (max_w - min_w) * 10
)

# 60% note hôtel, 40% météo (à ajuster selon les priorités du business)
df_merged["combined_score"] = (
    0.6 * df_merged["score"] + 0.4 * df_merged["weather_score_norm"]
).round(2)

df_merged.head()

,city,name,url,score,lat,lon,weather_score,temp_avg,rain_total,weather_score_norm,combined_score
0,Strasbourg,Budget Hôtel Weber - Strasbourg Centre Gare,https://www.booking.com/hotel/fr/weber.fr.html...,7.2,48.581243,7.732145,13.21,24.0,11.7,0.0,4.32
1,Strasbourg,Séjours & Affaires Strasbourg Kleber,https://www.booking.com/hotel/fr/residhome-str...,7.8,48.583197,7.742852,13.21,24.0,11.7,0.0,4.68
2,Strasbourg,City Résidence Strasbourg Centre,https://www.booking.com/hotel/fr/appart-victor...,7.8,48.589062,7.738799,13.21,24.0,11.7,0.0,4.68
3,Strasbourg,ibis Styles Strasbourg Avenue du Rhin,https://www.booking.com/hotel/fr/ibis-strasbou...,8.3,48.569366,7.777569,13.21,24.0,11.7,0.0,4.98
4,Strasbourg,Residence Inn by Marriott Strasbourg,https://www.booking.com/hotel/fr/residence-inn...,8.8,48.598995,7.762173,13.21,24.0,11.7,0.0,5.28


In [5]:
fig_combined = px.scatter_map(
    df_merged,
    lat="lat",
    lon="lon",
    color="combined_score",
    size="combined_score",
    size_max=18,
    hover_name="name",
    hover_data={
        "city": True,
        "score": ":.1f",
        "weather_score": ":.2f",
        "combined_score": ":.2f",
        "temp_avg": ":.1f",
        "rain_total": ":.1f",
        "weather_score_norm": False,
        "lat": False,
        "lon": False,
    },
    color_continuous_scale="RdYlGn",
    zoom=4.5,
    center={"lat": 46.6, "lon": 2.5},
    height=600,
    title="Score combiné météo + hôtel — recommandations de la semaine",
)
fig_combined.update_layout(map_style="carto-positron")
fig_combined.show()

## 4. Top 10 des recommandations

Vue tabulaire claire, qui peut servir de livrable texte au-delà de la carte.

In [6]:
top10 = (
    df_merged
    .nlargest(10, "combined_score")
    [["name", "city", "score", "weather_score", "combined_score", "url"]]
    .reset_index(drop=True)
)
top10

,name,city,score,weather_score,combined_score,url
0,Studio terrasse,Aix en Provence,9.9,27.59,9.92,https://www.booking.com/hotel/fr/studio-terras...
1,Maison Olea,Marseille,9.6,27.68,9.76,https://www.booking.com/hotel/fr/maison-olea-m...
2,La Résidence Du Vieux Port,Marseille,8.9,27.68,9.34,https://www.booking.com/hotel/fr/hotel-la-resi...
3,Maisons du Monde Hôtel & Suites - Marseille Vi...,Marseille,8.7,27.68,9.22,https://www.booking.com/hotel/fr/maisondumonde...
4,Hôtel Boutique Cézanne centre Aix-en-Provence,Aix en Provence,8.7,27.59,9.20,https://www.booking.com/hotel/fr/cezanne-aix-e...
5,Domaine Gao,Aix en Provence,8.7,27.59,9.20,https://www.booking.com/hotel/fr/domaine-and-c...
6,ibis Styles Marseille Palais des Congrès Vélod...,Marseille,8.6,27.68,9.16,https://www.booking.com/hotel/fr/ibis-styles-m...
7,Grand Hotel Beauvau Marseille Vieux-Port - MGa...,Marseille,8.6,27.68,9.16,https://www.booking.com/hotel/fr/hotel-1293-gr...
8,Hotel Cardinal,Aix en Provence,8.6,27.59,9.14,https://www.booking.com/hotel/fr/cardinal.fr.h...
9,Renaissance Aix-en-Provence Hotel,Aix en Provence,8.6,27.59,9.14,https://www.booking.com/hotel/fr/renaissance-a...


## 5. Sauvegardes finales

On exporte les datasets propres pour les parties suivantes (S3 et SQL).

In [7]:
# Hôtels propres (sans les valeurs manquantes de score)
df_hotels_scored.to_csv("data/hotels.csv", index=False)

# Dataset enrichi (hôtels + météo + score combiné)
df_merged.to_csv("data/recommendations.csv", index=False)

print("✅ data/hotels.csv         — 90 hôtels avec leurs notes Booking")
print("✅ data/recommendations.csv — 90 hôtels enrichis du score météo et combiné")

✅ data/hotels.csv         — 90 hôtels avec leurs notes Booking
✅ data/recommendations.csv — 90 hôtels enrichis du score météo et combiné
